# 08 — ESA Biomass P-band coverage feasibility (metadata only)

**Not yet processed P-band imagery.** Verify candidate scene footprints for Hakaluki Haor before downloading radar products or claiming cross-sensor wetland benefits. Source: [ESA Biomass MAAP](https://portal.maap.eo.esa.int/biomass/biomass-data-offer/).

In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

ROOT = Path(r"F:\Wetland Test\Hakaluki_Haor_Wetland_Test_Jupyter_GEE_Project")
BND = ROOT / "data" / "boundary" / "hakaluki_boundary.gpkg"
if not BND.exists():
    BND = ROOT / "Study area" / "Hakaluki_Haor.shp"
assert BND.exists(), "Run Notebook 00 to save the boundary or supply the original study shapefile."
haor = gpd.read_file(BND).to_crs("EPSG:4326")
print("Study polygon:", len(haor), "geometry valid:", haor.geometry.is_valid.all())

## A. Save ESA MAAP scene footprint metadata

Using an ESA-accessible catalogue, download the relevant imagery footprints in GeoJSON form and place the file here: `data/raw/biomass_scene_footprints.geojson`. This file should contain valid polygon footprints (product IDs/timestamps are recommended attributes). This notebook does not fetch or invent coverage.

In [ ]:
SCENES = ROOT / "data" / "raw" / "biomass_scene_footprints.geojson"
if not SCENES.exists():
    print("NOT AVAILABLE: ESA Biomass footprint GeoJSON not yet supplied.")
    print("See research_docs/BIOMASS_SCENE_FEASIBILITY.md")
else:
    scene = gpd.read_file(SCENES).to_crs("EPSG:4326")
    scene = scene.loc[scene.geometry.notna() & ~scene.geometry.is_empty].copy()
    if not scene.geometry.is_valid.all():
        print("WARNING: Invalid ESA scene footprint geometries. Review before interpretation.")
    scene["overlap_geom"] = scene.geometry.intersection(haor.geometry.unary_union)
    overlap = gpd.GeoSeries(scene["overlap_geom"],crs="EPSG:4326").to_crs("EPSG:32646")
    haor_km2 = haor.to_crs("EPSG:32646").geometry.area.sum()/1e6
    scene["overlap_km2"] = overlap.area/1e6
    scene["overlap_fraction_roi"] = scene["overlap_km2"] / haor_km2
    matches = scene.loc[scene["overlap_km2"] > 0].drop(columns=["overlap_geom"])
    out = ROOT / "results" / "tables" / "biomass_pband_footprint_feasibility.csv"
    out.parent.mkdir(parents=True, exist_ok=True)
    matches.drop(columns="geometry").to_csv(out,index=False)
    print("Intersecting scene footprints:",len(matches))
    print("Saved:",out)
    display(matches.head(20))

## B. Interpretation checklist

Footprint overlap does not prove usable P-band SAR acquisitions, wetland flooding sensitivity, suitable polarimetric products, or comparability with Sentinel-1. Review product type, quality, acquisition date, potential RF interference, and Hakaluki vegetation types before advancing to P-band classification. Record a negative or unavailable result explicitly if needed.